# 07 · Multi-Agent Orchestration: the Complete Research System

Everything comes together: the agents from notebooks 02–06, orchestrated with the LangGraph patterns from 05.

```
                 ┌──────────┐    ┌──────────────┐  Send × N   ┌──────────────┐
 query ─────────►│ planner  ├───►│ human_review ├────────────►│ researcher ×N│ (parallel)
                 └──────────┘    └──────────────┘ (interrupt) └──────┬───────┘
                                                                     │ findings (reducer)
                                    follow-up questions: Send × k    ▼
                 ┌──────────┐ no ┌──────────┐    ┌────────┐    ┌──────────────┐
                 │ finalize │◄───┤  critic  │◄───┤ writer │◄───┤ fact_checker │
                 └────┬─────┘ ok └────┬─────┘    └────────┘    └──────────────┘
                      ▼               └── gaps & iteration < max ──► researcher
                 report.md
```

### Engineering choices in this notebook
- **Dependency injection**: `build_research_graph(agents)` receives the agent functions. The real system passes the
  LLM agents; the error-injection test (§6) passes stubs, so it runs instantly and costs nothing.
- **Runtime configuration** via LangGraph's `config["configurable"]` (number of sub-questions, critic iterations,
  human review on/off): no code changes to tune a run.
- **Resilience**: researchers catch their own errors (Send tasks, see nb 05 §9c); sequential LLM nodes get a
  `RetryPolicy`; the fact-checker and critic have `error_handler`s that degrade gracefully; a **SQLite checkpointer**
  lets any crashed run resume from the failed step.
- **Observability**: per-node progress events (`custom` stream), timing metrics in state, and LangSmith traces.

In [ ]:
from nb_setup import setup, timer
ROOT = setup()

import operator, time
from typing import Annotated, TypedDict
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, Command, RetryPolicy, interrupt
from langgraph.errors import NodeError
from langgraph.config import get_stream_writer

from nb_components import ResearchPlan
from nb_agents import Agents, Finding, FactCheckReport, VerifiedClaim, Critique, finalize, credibility

## 1 · Shared state

The state is the **only** communication channel between agents. Keys written by parallel researchers
(`findings`, `failures`, `metrics`) use the `operator.add` reducer; everything else is written by one node at a time.

In [ ]:
class ResearchState(TypedDict, total=False):
    query: str
    plan: ResearchPlan
    findings: Annotated[list[Finding], operator.add]
    failures: Annotated[list[str], operator.add]
    warnings: Annotated[list[str], operator.add]
    metrics: Annotated[list[dict], operator.add]
    fact_check: FactCheckReport
    draft: str
    sources: dict[str, int]
    critique: Critique
    iteration: int
    report: str


def cfg(config: RunnableConfig, key: str, default):
    """Read a runtime setting from config['configurable']."""
    return config.get("configurable", {}).get(key, default)

## 2 · The graph builder

Each node is a thin wrapper: read what it needs from state → call its agent → return a partial update
(plus progress events and timing). All research logic lives in the agents; all control flow lives here.

In [ ]:
def build_research_graph(agents: Agents, checkpointer=None):
    llm_retry = RetryPolicy(max_attempts=2, initial_interval=2.0)

    # ---------------------------------------------------------------- nodes
    def planner(state: ResearchState, config: RunnableConfig) -> dict:
        get_stream_writer()({"agent": "planner", "status": "planning"})
        t0 = time.perf_counter()
        plan = agents.plan(state["query"])
        max_q = cfg(config, "max_sub_questions", 5)
        plan = plan.model_copy(update={"sub_questions": plan.sub_questions[:max_q]})
        return {"plan": plan, "iteration": 0,
                "metrics": [{"node": "planner", "seconds": time.perf_counter() - t0}]}

    def human_review(state: ResearchState, config: RunnableConfig) -> dict:
        if not cfg(config, "human_review", False):
            return {}
        decision = interrupt({"objective": state["plan"].objective,
                              "sub_questions": [q.question for q in state["plan"].sub_questions]})
        if edited := decision.get("sub_questions"):
            kept = [q for q in state["plan"].sub_questions if q.question in edited]
            return {"plan": state["plan"].model_copy(update={"sub_questions": kept or state["plan"].sub_questions})}
        return {}

    def researcher(payload: dict, config: RunnableConfig) -> dict:
        write = get_stream_writer()
        q = payload["sub_question"]
        write({"agent": "researcher", "status": "started", "sub_question": q})
        t0 = time.perf_counter()
        try:  # Send tasks bypass error_handler (nb 05 §9c), so we catch here
            finding = agents.research(q, payload.get("strategy", "web"),
                                      max_tool_calls=cfg(config, "max_tool_calls", 4))
            update = {"findings": [finding]}
            write({"agent": "researcher", "status": "done", "sub_question": q, "claims": len(finding.claims)})
        except Exception as e:
            update = {"failures": [f"{q} -> {type(e).__name__}: {str(e)[:120]}"]}
            write({"agent": "researcher", "status": "failed", "sub_question": q})
        update["metrics"] = [{"node": "researcher", "start": t0, "end": time.perf_counter(),
                              "seconds": time.perf_counter() - t0}]
        return update

    def fact_checker(state: ResearchState) -> dict:
        get_stream_writer()({"agent": "fact_checker", "status": f"checking {len(state['findings'])} findings"})
        t0 = time.perf_counter()
        fc = agents.fact_check(state["findings"])
        return {"fact_check": fc, "metrics": [{"node": "fact_checker", "seconds": time.perf_counter() - t0}]}

    def writer(state: ResearchState) -> dict:
        get_stream_writer()({"agent": "writer", "status": "writing report"})
        t0 = time.perf_counter()
        draft, sources = agents.write(state["plan"], state["fact_check"])
        return {"draft": draft, "sources": sources,
                "metrics": [{"node": "writer", "seconds": time.perf_counter() - t0}]}

    def critic(state: ResearchState) -> dict:
        get_stream_writer()({"agent": "critic", "status": "reviewing"})
        t0 = time.perf_counter()
        c = agents.critique(state["plan"], state["draft"])
        return {"critique": c, "iteration": state["iteration"] + 1,
                "metrics": [{"node": "critic", "seconds": time.perf_counter() - t0}]}

    def finalize_report(state: ResearchState) -> dict:
        report, problems = finalize(state["draft"], state["sources"])
        notes = [f"- Could not research: {f}" for f in state.get("failures", [])]
        notes += [f"- {w}" for w in state.get("warnings", []) + problems]
        if notes:  # dict.fromkeys de-duplicates (a node can fail in several critic rounds) and keeps order
            report += "\n## Limitations of this report\n" + "\n".join(dict.fromkeys(notes)) + "\n"
        return {"report": report}

    # ---------------------------------------------------------------- error handlers (graceful degradation)
    def fact_check_failed(state: ResearchState, error: NodeError) -> Command:
        # Skip the LLM conflict check: keep claims, mark them unchecked, continue to the writer.
        claims = [VerifiedClaim(statement=c.statement, source_urls=c.source_urls, status="supported",
                                credibility=max(credibility(u) for u in c.source_urls))
                  for f in state["findings"] for c in f.claims]
        return Command(update={"fact_check": FactCheckReport(claims=claims, conflicts=[]),
                               "warnings": [f"Fact-checking skipped ({type(error.error).__name__}); "
                                            "claims were not cross-verified."]}, goto="writer")

    def critic_failed(state: ResearchState, error: NodeError) -> Command:
        return Command(update={"warnings": [f"Quality review skipped ({type(error.error).__name__})."]},
                       goto="finalize")

    # ---------------------------------------------------------------- routing
    def dispatch_researchers(state: ResearchState) -> list[Send]:
        return [Send("researcher", {"sub_question": q.question, "strategy": q.search_strategy})
                for q in state["plan"].sub_questions]

    def after_critic(state: ResearchState, config: RunnableConfig):
        c = state["critique"]
        if (c.approved(cfg(config, "approval_threshold", 4))
                or state["iteration"] >= cfg(config, "max_iterations", 2)
                or not c.follow_up_questions):
            return "finalize"
        return [Send("researcher", {"sub_question": q, "strategy": "web"}) for q in c.follow_up_questions]

    # ---------------------------------------------------------------- wiring
    g = StateGraph(ResearchState)
    g.add_node("planner", planner, retry_policy=llm_retry)
    g.add_node("human_review", human_review)
    g.add_node("researcher", researcher)
    g.add_node("fact_checker", fact_checker, retry_policy=llm_retry, error_handler=fact_check_failed)
    g.add_node("writer", writer, retry_policy=llm_retry)
    g.add_node("critic", critic, retry_policy=llm_retry, error_handler=critic_failed)
    g.add_node("finalize", finalize_report)

    g.add_edge(START, "planner")
    g.add_edge("planner", "human_review")
    g.add_conditional_edges("human_review", dispatch_researchers, ["researcher"])
    g.add_edge("researcher", "fact_checker")
    g.add_edge("fact_checker", "writer")
    g.add_edge("writer", "critic")
    g.add_conditional_edges("critic", after_critic, ["researcher", "finalize"])
    g.add_edge("finalize", END)
    return g.compile(checkpointer=checkpointer)


print(build_research_graph(Agents()).get_graph().draw_mermaid())

## 3 · A persistent checkpointer: surviving crashes and quota exhaustion

**What happened on our first attempt:** with an in-memory checkpointer, the run got through planning, plan approval,
3 parallel researchers and fact-checking, and then the Writer hit OpenRouter's
`free-models-per-day` limit (50 requests/day without credits; every free model shares it, so fallbacks can't help).
The whole run failed and **all completed research was lost** with the kernel.

The fix is to checkpoint to **SQLite**. LangGraph saves state after every step, so after any crash, calling
`graph.invoke(None, config)` on the same `thread_id` **resumes at the failed node**, even in a new process the next
day. Completed researchers are not re-run (we verified this with stub agents: 3 researcher calls in total, not 6).

**Security detail:** checkpoints are deserialised from disk. Instead of letting LangGraph load *any* Python type
(it warns about this), we pass an explicit **allowlist** of our own schema classes.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from nb_components import SubQuestion
from nb_agents import Claim

SAFE_TYPES = [ResearchPlan, SubQuestion, Finding, Claim, FactCheckReport, VerifiedClaim, Critique]
checkpointer = SqliteSaver(sqlite3.connect(ROOT / "data" / "07_checkpoints.sqlite", check_same_thread=False),
                           serde=JsonPlusSerializer(allowed_msgpack_modules=SAFE_TYPES))
research_graph = build_research_graph(Agents(), checkpointer=checkpointer)

## 4 · Run it: streaming progress + human-in-the-loop

Settings for this run (kept small for the free tier, since one run uses roughly 20–35 LLM requests):
3 sub-questions, ≤3 tool calls per researcher, at most 2 critic iterations, human review **on**.

`advance()` moves the thread forward from wherever it is, which makes this notebook **safe to re-run**:
- no checkpoint yet → start a new run with the query;
- paused at the `interrupt` → resume with the human's decision;
- crashed mid-run (e.g. quota) → resume from the failed node with `None`;
- finished → nothing to do.

The first call runs until the `interrupt` in `human_review` and shows the plan for approval.

In [ ]:
QUERY = "How are AI coding agents changing software engineering workflows in 2026?"
RUN_CONFIG = {"configurable": {"thread_id": "demo-1", "max_sub_questions": 3, "max_tool_calls": 3,
                               "max_iterations": 2, "human_review": True},
              "recursion_limit": 40}


def show(events):
    for mode, event in events:
        if mode == "custom":
            print(f"  [{event['agent']}] {event['status']}" + (f": {event['sub_question'][:70]}" if "sub_question" in event else ""))
        elif mode == "updates" and "__interrupt__" in event:
            print("\n>>> PAUSED for human review:")
            for q in event["__interrupt__"][0].value["sub_questions"]:
                print("    -", q)

def advance(human_decision: dict | None = None):
    """Move the run forward from its current checkpoint (start, approve, or resume after a crash)."""
    snap = research_graph.get_state(RUN_CONFIG)
    if snap.interrupts:
        inp = Command(resume=human_decision or {"approved": True})
    elif snap.next:
        print(f"Resuming unfinished run at {snap.next} (completed steps are not repeated)")
        inp = None
    elif snap.values.get("report"):
        print("This thread already finished. Use a new thread_id to run again.")
        return
    else:
        inp = {"query": QUERY}
    t0 = time.perf_counter()
    try:
        show(research_graph.stream(inp, RUN_CONFIG, stream_mode=["custom", "updates"]))
    except Exception as e:
        print(f"\n!!! Run stopped: {type(e).__name__}: {str(e)[:150]}")
        print(f"    Pending node: {research_graph.get_state(RUN_CONFIG).next}. Re-run advance() later to resume.")
    print(f"[{time.perf_counter() - t0:.0f}s]")


advance()   # start -> runs until human review

The graph is now **paused and checkpointed**. A UI would show the plan with checkboxes; here we approve it as-is
by resuming the same `thread_id` with `Command(resume=...)`. Researchers then run in parallel.

In [ ]:
advance({"approved": True})   # approve -> research, fact-check, write, critique (loop) -> finalize
final = research_graph.get_state(RUN_CONFIG).values
assert final.get("report"), "Run not finished yet (see message above): re-run this cell later to resume."

## 5 · The report

In [ ]:
from IPython.display import Markdown, display

display(Markdown(final["report"]))

In [ ]:
# Save the report as an artifact of this experiment
out = ROOT / "reports" / "07_demo_report.md"
out.write_text(final["report"], encoding="utf-8")
print("saved ->", out.relative_to(ROOT))

## 6 · Run metrics

What did the run cost, and did parallelism pay off? *Sequential time* is the sum of all researcher durations, i.e.
what one-after-another research would have taken; the *parallel* figure is the wall-clock span of each research
round. This gives the speedup without paying for a second, sequential run.

In [ ]:
import pandas as pd

m = pd.DataFrame(final["metrics"])
researchers = m[m.node == "researcher"]
rounds = max(final["iteration"], 1)
c = final["critique"]

print(f"critic iterations : {final['iteration']}  (approved={c.approved()}; scores "
      f"coverage={c.coverage}, evidence={c.evidence_use}, coherence={c.coherence})")
print(f"findings / failed : {len(final['findings'])} / {len(final.get('failures', []))}")
fc = final["fact_check"]
print(f"claims            : " + ", ".join(f"{s}={sum(x.status == s for x in fc.claims)}"
                                          for s in ("supported", "contested", "unverified")))
print(f"conflicts found   : {len(fc.conflicts)}")
print(f"sources cited     : {len(final['sources'])}")
seq = researchers.seconds.sum()
span = researchers.end.max() - researchers.start.min()   # includes the gap between rounds if the critic looped
print(f"research time     : sequential≈{seq:.0f}s vs actual {span:.0f}s across {rounds} round(s)")
m.groupby("node").seconds.agg(["count", "sum", "mean"]).round(1)

## 7 · Error injection: does the system degrade gracefully?

We rebuild the graph with **stub agents** (thanks to dependency injection; no LLM calls, runs in milliseconds)
and inject three failures at once:
1. one researcher raises (simulated search outage),
2. the fact-checker raises (→ `error_handler` skips it with a warning),
3. the critic rejects the first draft with a follow-up question (→ the loop runs a second research round).

Expected: a complete report, a "Limitations" section listing the failed sub-question and skipped fact-check, and
two critic iterations. These stubs are exactly what the Phase 3 integration tests will use.

In [ ]:
from nb_components import SubQuestion
from nb_agents import Claim, write as real_write

def stub_plan(query):
    return ResearchPlan(objective=f"Study {query}", research_type="exploratory", report_outline=["Intro", "Findings", "Outlook"],
                        sub_questions=[SubQuestion(question=q, rationale="r", search_strategy="web")
                                       for q in ("costs", "outage-prone topic", "adoption")])

def stub_research(q, strategy="web", **_):
    if q == "outage-prone topic":
        raise ConnectionError("search provider down")
    url = f"https://example.org/{q.replace(' ', '-')}"
    return Finding(sub_question=q, summary=f"About {q}", confidence="medium", retrieved_urls=[url],
                   claims=[Claim(statement=f"Key fact about {q}.", source_urls=[url])])

def broken_fact_check(findings):
    raise TimeoutError("fact-check model timed out")

def stub_write(plan, fc):
    numbers = {u: i for i, u in enumerate(dict.fromkeys(u for c in fc.claims for u in c.source_urls), 1)}
    body = " ".join(f"{c.statement} [{numbers[c.source_urls[0]]}]" for c in fc.claims)
    return f"# Stub report\n\n{body}", numbers

critic_calls = {"n": 0}
def stub_critique(plan, report):
    critic_calls["n"] += 1
    if critic_calls["n"] == 1:
        return Critique(coverage=2, evidence_use=3, coherence=4, justification="Missing pricing.",
                        gaps=["pricing"], follow_up_questions=["pricing models"])
    return Critique(coverage=4, evidence_use=4, coherence=4, justification="Good enough.")

stub_graph = build_research_graph(Agents(plan=stub_plan, research=stub_research, fact_check=broken_fact_check,
                                         write=stub_write, critique=stub_critique))
t0 = time.perf_counter()
out = stub_graph.invoke({"query": "widgets"}, {"configurable": {"max_iterations": 2}})
print(f"ran in {time.perf_counter() - t0:.2f}s | critic iterations: {out['iteration']} | findings: {len(out['findings'])}\n")
print(out["report"])

## 8 · Observations & Decisions

*Recorded from the run on 2026-11-19; free-model runs vary.*

**Observations**
- See §5 for this run's numbers: critic iterations, claim statuses, conflicts, sources, per-node timings and the
  sequential-vs-parallel research time.
- The first attempt exhausted the free daily quota at the Writer step and lost all research (in-memory checkpoints).
  With `SqliteSaver`, the same failure becomes a pause: `advance()` resumes at the failed node without re-running
  completed researchers.
- Human-in-the-loop works across two separate `stream` calls because the state is checkpointed under a `thread_id`.
- The error-injection run completed in well under a second with a full report: one failed researcher, a skipped
  fact-check and a critic-triggered second research round were all handled and surfaced in "Limitations".
- Dependency injection made the failure test free and deterministic: the same pattern will drive Phase 3 tests.

**Decisions** (for the Phase 2 implementation)
1. Graph topology and node responsibilities exactly as built here → `src/research_system/graph/builder.py`.
2. `Agents` dependency container → the graph never imports concrete LLM code; tests inject stubs.
3. Runtime knobs via `config["configurable"]` (`max_sub_questions`, `max_tool_calls`, `max_iterations`,
   `approval_threshold`, `human_review`) + `recursion_limit` safety net.
4. Error policy: researchers catch-and-record; planner/writer retry then fail; fact-checker/critic retry then degrade.
   Use an **explicit** `retry_on`: LangGraph's default deliberately skips `RuntimeError`/`ValueError`/`OSError` (incl.
   `TimeoutError`) and retries unknown types, so per-minute 429s should be retried but daily-quota errors should not.
5. The report always ends with code-built References and, when relevant, a "Limitations" section.
6. Production checkpointer: `SqliteSaver` with a type **allowlist** serializer; streaming events (`custom` + `updates`) → FastAPI SSE in Phase 4.